# Neural Gradient Boosting — Credit Risk Default Prediction

**Gradient boosting with weak neural networks as base learners.**

### Key principles
- **Weak learners** — intentionally small MLPs (`[32, 16]`), too weak to overfit alone
- **Low learning rate** — `lr=0.1`, many estimators (50–100) for gradual correction
- **MSE on residuals** — each learner fits the residual error of the ensemble so far
- **Early stopping** — per-learner early stopping, plus global early stopping on val loss

### Comparison
We evaluate against the single MLP and stacking ensemble from the other notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    precision_recall_curve, RocCurveDisplay, classification_report
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# =============================================================================
# Environment setup — works both locally and in Google Colab
# =============================================================================
try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected.")
    print("Upload train.csv, val.csv, test.csv to /content/ or set DATA_PATH below.")
    DATA_PATH = "/content"
except ImportError:
    IN_COLAB = False
    DATA_PATH = "../../data/processed"
    print("Local environment detected.")

OUT_DIR = "./output"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUT_DIR:   {OUT_DIR}")

In [ ]:
# --- Load data ---
train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
val_df   = pd.read_csv(f"{DATA_PATH}/val.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")

full_train_df = pd.concat([train_df, val_df], ignore_index=True)

X_full = full_train_df.drop(columns=['DEFAULT_OCT']).values.astype(np.float32)
y_full = full_train_df['DEFAULT_OCT'].values.astype(np.float32)

X_test = test_df.drop(columns=['DEFAULT_OCT']).values.astype(np.float32)
y_test = test_df['DEFAULT_OCT'].values.astype(np.float32)

# Hold-out val split from full_train for early stopping
X_tr, X_es, y_tr, y_es = train_test_split(
    X_full, y_full, test_size=0.15, stratify=y_full, random_state=42
)

input_dim = X_full.shape[1]
print(f"Train: {X_tr.shape[0]:,}  |  ES-val: {X_es.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Input dim: {input_dim}  |  Default rate: {y_full.mean():.4f}")

In [ ]:
# =============================================================================
# WeakMLP — intentionally small, kept weak to serve as a boosting base learner
# =============================================================================

class WeakMLP(nn.Module):
    """A deliberately small MLP for gradient boosting.
    - 2 hidden layers, small width [32, 16]
    - Dropout to prevent any single learner from overfitting
    - No output activation → fits raw residuals (can be negative or positive)
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# Quick param count
dummy = WeakMLP(input_dim)
print(f"WeakMLP parameters: {sum(p.numel() for p in dummy.parameters()):,}")

In [ ]:
# =============================================================================
# BoostedMLPClassifier — gradient boosting with neural weak learners
# =============================================================================

class BoostedMLPClassifier:
    def __init__(self, input_dim, n_estimators=100, learning_rate=0.1,
                 batch_size=512, epochs_per_estimator=30, patience_per_estimator=8,
                 global_patience=15, minority_weight=3.5):
        self.input_dim = input_dim
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs_per_estimator = epochs_per_estimator
        self.patience_per_estimator = patience_per_estimator
        self.global_patience = global_patience
        self.minority_weight = minority_weight

        self.ensemble = nn.ModuleList()
        self.initial_log_odds = 0.0
        self.train_losses_ = []
        self.es_losses_ = []

    def _compute_residuals(self, logits, y):
        """Compute pseudo-residuals: gradient of BCE loss w.r.t. logits.
        d/dz BCE(z, y) = sigmoid(z) - y  →  residual = y - p
        Positive residual → model underpredicts default → next learner adds to logit.
        """
        p = torch.sigmoid(torch.tensor(logits, dtype=torch.float32))
        return y - p.numpy()

    def fit(self, X_train, y_train, X_es=None, y_es=None, verbose=True):
        """Fit the boosted ensemble."""
        n_samples = X_train.shape[0]

        # Initial prediction: log-odds of the training prior
        pos_rate = y_train.mean()
        self.initial_log_odds = np.log(pos_rate / (1 - pos_rate))
        current_logits = np.full(n_samples, self.initial_log_odds, dtype=np.float32)

        if X_es is not None:
            es_logits = np.full(X_es.shape[0], self.initial_log_odds, dtype=np.float32)

        best_es_loss = float('inf')
        best_ensemble_state = None
        global_counter = 0

        for est_idx in range(self.n_estimators):
            # --- Compute residuals ---
            residuals = self._compute_residuals(current_logits, y_train)

            # --- Train a WeakMLP on residuals using WEIGHTED MSE ---
            # Minority-class residuals get higher weight (default: 3.5×).
            # This forces each weak learner to prioritise correcting errors
            # on the defaulting class, which is where most signal lives.
            weak = WeakMLP(self.input_dim).to(device)
            optimizer = optim.Adam(weak.parameters(), lr=1e-3, weight_decay=1e-5)

            X_t = torch.tensor(X_train, dtype=torch.float32)
            r_t = torch.tensor(residuals, dtype=torch.float32)
            y_t = torch.tensor(y_train, dtype=torch.float32)
            train_ds = TensorDataset(X_t, r_t, y_t)
            train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)

            best_learner_loss = float('inf')
            best_learner_state = None
            learner_counter = 0

            for epoch in range(self.epochs_per_estimator):
                weak.train()
                epoch_loss = 0.0
                for Xb, rb, yb in train_loader:
                    Xb, rb, yb = Xb.to(device), rb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = weak(Xb)
                    # Weighted MSE: minority-class residuals get
                    # minority_weight × the loss of majority-class residuals
                    weights = torch.where(
                        yb == 1,
                        torch.tensor(self.minority_weight, device=device),
                        torch.tensor(1.0, device=device)
                    )
                    loss = (weights * (pred - rb) ** 2).mean()
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item() * Xb.size(0)
                epoch_loss /= n_samples

                if epoch_loss < best_learner_loss:
                    best_learner_loss = epoch_loss
                    best_learner_state = deepcopy(weak.state_dict())
                    learner_counter = 0
                else:
                    learner_counter += 1

                if learner_counter >= self.patience_per_estimator:
                    break

            weak.load_state_dict(best_learner_state)
            self.ensemble.append(weak)

            # --- Update ensemble predictions ---
            weak.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
                update = weak(X_tensor).cpu().numpy()
            current_logits += self.learning_rate * update

            # --- Evaluate on training set ---
            train_probs = 1.0 / (1.0 + np.exp(-current_logits))
            train_loss = -np.mean(
                y_train * np.log(np.clip(train_probs, 1e-15, 1)) +
                (1 - y_train) * np.log(np.clip(1 - train_probs, 1e-15, 1))
            )
            self.train_losses_.append(train_loss)

            # --- Evaluate on early-stopping set ---
            if X_es is not None:
                with torch.no_grad():
                    X_es_t = torch.tensor(X_es, dtype=torch.float32).to(device)
                    es_update = weak(X_es_t).cpu().numpy()
                es_logits += self.learning_rate * es_update

                es_probs = 1.0 / (1.0 + np.exp(-es_logits))
                es_loss = -np.mean(
                    y_es * np.log(np.clip(es_probs, 1e-15, 1)) +
                    (1 - y_es) * np.log(np.clip(1 - es_probs, 1e-15, 1))
                )
                self.es_losses_.append(es_loss)

                # Global early stopping
                if es_loss < best_es_loss:
                    best_es_loss = es_loss
                    best_ensemble_state = [deepcopy(m.state_dict()) for m in self.ensemble]
                    global_counter = 0
                else:
                    global_counter += 1

            if verbose and (est_idx + 1) % 10 == 0:
                es_str = f" | es_loss: {self.es_losses_[-1]:.4f}" if X_es is not None else ""
                print(f"  Estimator {est_idx+1:3d}/{self.n_estimators} | train_bce: {train_loss:.4f}{es_str}")

            if X_es is not None and global_counter >= self.global_patience:
                if verbose:
                    print(f"\n  Global early stopping at estimator {est_idx+1} (best: {est_idx+1-global_counter})")
                break

        # Restore best ensemble
        if X_es is not None and best_ensemble_state is not None:
            for m, state in zip(self.ensemble, best_ensemble_state):
                m.load_state_dict(state)
            self.ensemble = nn.ModuleList(list(self.ensemble)[:len(best_ensemble_state)])

        self.n_estimators_fitted_ = len(self.ensemble)
        return self

    def predict_logits(self, X):
        """Return raw logits (before sigmoid)."""
        n = X.shape[0]
        logits = np.full(n, self.initial_log_odds, dtype=np.float32)
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        for weak in self.ensemble:
            weak.eval()
            with torch.no_grad():
                logits += self.learning_rate * weak(X_tensor).cpu().numpy()
        return logits

    def predict_proba(self, X):
        """Return probability of class 1 (default)."""
        logits = self.predict_logits(X)
        return 1.0 / (1.0 + np.exp(-logits))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("BoostedMLPClassifier defined (with weighted MSE for minority class).")

In [ ]:
# =============================================================================
# Optuna Tuning — Boosting rate, weak-learner size, early stopping
# =============================================================================
RUN_TUNING = True
N_TRIALS   = 25

if RUN_TUNING:
    import optuna
    from optuna.samplers import TPESampler

    def objective(trial):
        lr_boost       = trial.suggest_float('lr_boost', 0.03, 0.3, log=True)
        width          = trial.suggest_categorical('weak_width', [16, 32, 64])
        epochs_per_est = trial.suggest_int('epochs_per_est', 10, 50)
        patience_est   = trial.suggest_int('patience_est', 3, 15)
        lr_learner     = trial.suggest_float('lr_learner', 5e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical('batch_size', [256, 512])

        # Build WeakMLP with tuned width
        class TunedWeakMLP(nn.Module):
            def __init__(self, input_dim):
                super().__init__()
                w = width
                self.net = nn.Sequential(
                    nn.Linear(input_dim, w), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(w, w//2), nn.ReLU(), nn.Dropout(0.15),
                    nn.Linear(w//2, 1)
                )
            def forward(self, x):
                return self.net(x).squeeze(-1)

        # Minimal boosted ensemble
        pos_rate = y_tune_tr.mean()
        logits = np.full(len(y_tune_tr), np.log(pos_rate/(1-pos_rate)), dtype=np.float32)
        es_logits = np.full(len(y_tune_es), np.log(pos_rate/(1-pos_rate)), dtype=np.float32)

        ensemble = nn.ModuleList()
        best_es, counter = float('inf'), 0

        for est_idx in range(80):
            p = 1.0/(1.0+np.exp(-logits))
            residuals = (y_tune_tr - p).astype(np.float32)

            weak = TunedWeakMLP(input_dim).to(device)
            opt = optim.Adam(weak.parameters(), lr=lr_learner, weight_decay=1e-5)
            ds = TensorDataset(torch.tensor(X_tune_tr, dtype=torch.float32),
                               torch.tensor(residuals, dtype=torch.float32))
            loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

            best_mse, mse_counter = float('inf'), 0
            best_state = None
            for ep in range(epochs_per_est):
                weak.train()
                for Xb, rb in loader:
                    Xb, rb = Xb.to(device), rb.to(device)
                    opt.zero_grad()
                    l = nn.functional.mse_loss(weak(Xb), rb)
                    l.backward(); opt.step()
                weak.eval()
                with torch.no_grad():
                    mse = nn.functional.mse_loss(weak(torch.tensor(X_tune_tr, dtype=torch.float32).to(device)),
                                                 torch.tensor(residuals, dtype=torch.float32).to(device)).item()
                if mse < best_mse:
                    best_mse, best_state = mse, deepcopy(weak.state_dict())
                    mse_counter = 0
                else:
                    mse_counter += 1
                if mse_counter >= patience_est:
                    break
            weak.load_state_dict(best_state)
            ensemble.append(weak)

            weak.eval()
            with torch.no_grad():
                logits += lr_boost * weak(torch.tensor(X_tune_tr, dtype=torch.float32).to(device)).cpu().numpy()
                es_logits += lr_boost * weak(torch.tensor(X_tune_es, dtype=torch.float32).to(device)).cpu().numpy()

            es_probs = 1.0/(1.0+np.exp(-es_logits))
            es_loss = -np.mean(y_tune_es*np.log(np.clip(es_probs,1e-15,1)) +
                               (1-y_tune_es)*np.log(np.clip(1-es_probs,1e-15,1)))

            trial.report(es_loss, est_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

            if es_loss < best_es:
                best_es, counter = es_loss, 0
            else:
                counter += 1
            if counter >= 20:
                break

        # Return AUC instead of loss (maximize)
        es_probs = 1.0/(1.0+np.exp(-es_logits))
        return roc_auc_score(y_tune_es, es_probs)

    # Sub-sample for faster tuning
    X_tune_tr, X_tune_es, y_tune_tr, y_tune_es = train_test_split(
        X_tr, y_tr, test_size=0.20, stratify=y_tr, random_state=42
    )
    print(f"Tuning neural boosting ({N_TRIALS} trials, {X_tune_tr.shape[0]:,} train / {X_tune_es.shape[0]:,} es)...\n")

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest trial #{study.best_trial.number}")
    print(f"Best val AUC: {study.best_value:.4f}")
    for k, v in study.best_params.items():
        print(f"  {k:20s} = {v}")

    BEST_BOOST_PARAMS = study.best_params
else:
    print("Tuning skipped. Using defaults.")
    BEST_BOOST_PARAMS = {}

# Hyperparameter Tuning with Optuna

Tune boosting rate, weak-learner architecture, and early-stopping settings.

In [ ]:
# =============================================================================
# Train the Boosted MLP — with tuned or default hyperparameters
# =============================================================================

if BEST_BOOST_PARAMS:
    boosted = BoostedMLPClassifier(
        input_dim=input_dim,
        n_estimators=100,
        learning_rate=BEST_BOOST_PARAMS['lr_boost'],
        batch_size=BEST_BOOST_PARAMS['batch_size'],
        epochs_per_estimator=BEST_BOOST_PARAMS['epochs_per_est'],
        patience_per_estimator=BEST_BOOST_PARAMS['patience_est'],
        global_patience=20
    )
    print(f"Using tuned params: lr={BEST_BOOST_PARAMS['lr_boost']:.3f}, "
          f"width={BEST_BOOST_PARAMS['weak_width']}, batch={BEST_BOOST_PARAMS['batch_size']}")
else:
    boosted = BoostedMLPClassifier(
        input_dim=input_dim, n_estimators=100, learning_rate=0.1,
        batch_size=512, epochs_per_estimator=30,
        patience_per_estimator=8, global_patience=15
    )
    print("Using default hyperparameters")

print("Training BoostedMLPClassifier...\n")
boosted.fit(X_tr, y_tr, X_es, y_es, verbose=True)
print(f"\nFitted {boosted.n_estimators_fitted_} estimators (early-stopped from {boosted.n_estimators})")

In [ ]:
# --- Training curves ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(boosted.train_losses_, label='Train BCE Loss', alpha=0.7)
ax.plot(boosted.es_losses_, label='ES-val BCE Loss', alpha=0.7)
ax.axvline(x=boosted.n_estimators_fitted_ - 1, color='gray', linestyle='--',
           alpha=0.5, label=f'Early stop ({boosted.n_estimators_fitted_} estimators)')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Neural Boosting: Loss per Round')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Evaluate on test set
# =============================================================================

boosted_probs = boosted.predict_proba(X_test)
boosted_preds = (boosted_probs >= 0.5).astype(int)

auc_boost = roc_auc_score(y_test, boosted_probs)
ap_boost  = average_precision_score(y_test, boosted_probs)
brier     = brier_score_loss(y_test, boosted_probs)

print(f"{'='*50}")
print(f"Neural Boosting — Test Set Results")
print(f"{'='*50}")
print(f"  AUC-ROC:       {auc_boost:.4f}")
print(f"  Avg Precision: {ap_boost:.4f}")
print(f"  Brier Score:   {brier:.4f}")
print(f"  Precision:     {precision_score(y_test, boosted_preds):.4f}")
print(f"  Recall:        {recall_score(y_test, boosted_preds):.4f}")
print(f"  F1 Score:      {f1_score(y_test, boosted_preds):.4f}")
print(f"\n{classification_report(y_test, boosted_preds, target_names=['Non-Default', 'Default'])}")

In [ ]:
# =============================================================================
# Threshold Tuning — optimise F1 on val set, evaluate on test
# =============================================================================
# Default threshold (0.5) often gives low recall on imbalanced data.
# Find the threshold that maximises F1 using the early-stopping (val) set,
# then apply it to the test set.

# Get predictions on the ES-val set (used during training)
es_probs = boosted.predict_proba(X_es)

# Sweep thresholds
thresholds = np.linspace(0.10, 0.80, 100)
f1_scores = []
for t in thresholds:
    preds = (es_probs >= t).astype(int)
    f1_scores.append(f1_score(y_es, preds, zero_division=0))

best_t = thresholds[np.argmax(f1_scores)]
best_f1_val = max(f1_scores)

# Apply tuned threshold to test set
boosted_preds_tuned = (boosted_probs >= best_t).astype(int)

print(f"Optimal threshold: {best_t:.2f} (val F1 = {best_f1_val:.4f})")
print(f"\n--- Default threshold (0.5) ---")
print(f"  Precision: {precision_score(y_test, boosted_preds):.4f}")
print(f"  Recall:    {recall_score(y_test, boosted_preds):.4f}")
print(f"  F1:        {f1_score(y_test, boosted_preds):.4f}")
print(f"\n--- Tuned threshold ({best_t:.2f}) ---")
print(f"  Precision: {precision_score(y_test, boosted_preds_tuned):.4f}")
print(f"  Recall:    {recall_score(y_test, boosted_preds_tuned):.4f}")
print(f"  F1:        {f1_score(y_test, boosted_preds_tuned):.4f}")
print(f"\n{classification_report(y_test, boosted_preds_tuned, target_names=['Non-Default', 'Default'])}")

# Use tuned predictions going forward
boosted_preds = boosted_preds_tuned

In [ ]:
# =============================================================================
# Visual evaluation
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- ROC Curve ---
RocCurveDisplay.from_predictions(y_test, boosted_probs, ax=axes[0, 0])
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 0].set_title(f'ROC Curve — Neural Boosting (AUC = {auc_boost:.4f})')

# --- Precision-Recall Curve ---
prec, rec, _ = precision_recall_curve(y_test, boosted_probs)
axes[0, 1].plot(rec, prec, color='#2ecc71', linewidth=2)
axes[0, 1].axhline(y=y_test.mean(), color='gray', linestyle=':', alpha=0.5,
                   label=f'Baseline ({y_test.mean():.2f})')
axes[0, 1].fill_between(rec, prec, alpha=0.1, color='#2ecc71')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title(f'PR Curve (AP = {ap_boost:.4f})')
axes[0, 1].legend(loc='upper right')

# --- Probability distribution by class ---
for label, color, name in [(0, '#3498db', 'Non-Default'), (1, '#e74c3c', 'Default')]:
    mask = y_test == label
    sns.kdeplot(boosted_probs[mask], fill=True, alpha=0.3, color=color,
                label=name, ax=axes[1, 0])
axes[1, 0].axvline(x=0.5, color='gray', linestyle='--', label='Threshold = 0.5')
axes[1, 0].set_xlabel('Predicted Probability')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Probability Distribution by Actual Class')
axes[1, 0].legend()

# --- Confusion Matrix ---
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, boosted_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Individual estimator contribution — how predictions evolve per round
# =============================================================================

# Track logits after each estimator for a few test samples
sample_indices = [0, 1, 2, 3, 4]  # first 5 test samples
logits_over_time = np.zeros((boosted.n_estimators_fitted_, len(sample_indices)))

X_sample = X_test[sample_indices]
running = np.full(len(sample_indices), boosted.initial_log_odds, dtype=np.float32)

X_sample_t = torch.tensor(X_sample, dtype=torch.float32).to(device)
for i, weak in enumerate(boosted.ensemble):
    weak.eval()
    with torch.no_grad():
        running += boosted.learning_rate * weak(X_sample_t).cpu().numpy()
    logits_over_time[i] = running

probs_over_time = 1.0 / (1.0 + np.exp(-logits_over_time))

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(sample_indices)))
for i in range(len(sample_indices)):
    true_label = y_test[sample_indices[i]]
    ax.plot(probs_over_time[:, i], color=colors[i],
            label=f'Sample {sample_indices[i]} (true={int(true_label)})', linewidth=1.5)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold = 0.5')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Predicted Probability of Default')
ax.set_title('How Predictions Evolve as More Estimators Are Added')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Compare: Neural Boosting vs Single MLP (from model-building-test.ipynb)
# =============================================================================

print("Comparison Summary:")
print("="*55)
print(f"  Neural Boosting  — AUC: {auc_boost:.4f}, AP: {ap_boost:.4f}, Brier: {brier:.4f}")

# Try to load the single MLP for comparison
single_mlp_path = "../../data/processed/best_model.pt"
import os
if os.path.exists(single_mlp_path):
    # Define same architecture as the single MLP
    class SingleMLP(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.backbone = nn.Sequential(
                nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(32, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(0.2),
            )
            self.head = nn.Linear(16, 1)
        def forward(self, x):
            return self.head(self.backbone(x)).squeeze(-1)

    single = SingleMLP(input_dim).to(device)
    single.load_state_dict(torch.load(single_mlp_path, weights_only=True))
    single.eval()

    with torch.no_grad():
        single_logits = single(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
    single_probs = 1.0 / (1.0 + np.exp(-single_logits))

    auc_single = roc_auc_score(y_test, single_probs)
    ap_single  = average_precision_score(y_test, single_probs)
    brier_single = brier_score_loss(y_test, single_probs)

    print(f"  Single MLP       — AUC: {auc_single:.4f}, AP: {ap_single:.4f}, Brier: {brier_single:.4f}")
    print(f"\n  Boosting Δ AUC:   {auc_boost - auc_single:+.4f}")
    print(f"  Boosting Δ AP:    {ap_boost - ap_single:+.4f}")
    print(f"  Boosting Δ Brier: {brier_single - brier_boost:+.4f}")
else:
    print("  (Single MLP model not found — run model-building-test.ipynb first)")

print("="*55)

In [ ]:
# --- Save model ---
import joblib

torch.save(
    [m.state_dict() for m in boosted.ensemble],
    f"{OUT_DIR}/ensemble.pt"
)

config = {
    'input_dim': input_dim,
    'n_estimators': boosted.n_estimators_fitted_,
    'learning_rate': boosted.learning_rate,
    'initial_log_odds': boosted.initial_log_odds,
}
with open(f"{OUT_DIR}/config.pkl", 'wb') as f:
    pickle.dump(config, f)

# Save metrics
pd.DataFrame({
    'auc_roc': [auc_boost], 'avg_precision': [ap_boost], 'brier': [brier]
}).to_csv(f"{OUT_DIR}/results.csv", index=False)

print(f"Models saved to {OUT_DIR}/")
print(f"  ensemble.pt — {boosted.n_estimators_fitted_} WeakMLP state dicts")
print(f"  config.pkl, results.csv")

if IN_COLAB:
    from google.colab import files
    import shutil
    shutil.make_archive(f"{OUT_DIR}/models", 'zip', OUT_DIR)
    print(f"\nDownloading models.zip...")
    files.download(f"{OUT_DIR}/models.zip")